In [ ]:
import time
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0)
chip.add_compiler("./code/")
chip.adc.set_gap(adc_cs_gap=200,adc_first_gap=10,adc_last_gap=10)

In [ ]:
# chip.adc.set_gain_resistor(big_resistance=33e3,small_resistance=200)

# 拟合直方图

In [ ]:
from scipy.stats import norm
from scipy.optimize import curve_fit

In [ ]:
states = np.array([i*25+225 for i in range(32)])
root_path = "./result/4bit精度多次写验证结果/"
write_times = 1

cond_sub_base_path = []
for i in states:
    cond_sub_base_path.append(root_path+f"{i}us/after_write_verify_time={write_times}.npy")


# 初始化正态分布的预测情况
initial_guess = [
    # [225, 100, 900, 250, 1, 1],
]
for i in range(32):
    initial_guess.append([states[i],100,900,250,1,1])

def bimodal(x, mu1, sigma1, mu2, sigma2, A1, A2):
    return A1 * norm.pdf(x, mu1, sigma1) + A2 * norm.pdf(x, mu2, sigma2)

def normal_distribution(x, mu, sigma, A):
    return A * norm.pdf(x, mu, sigma)

# 设置直方图的区间
interval = 1
vmax = 1500
bin_edges = np.linspace(0, vmax, int(vmax / interval) + 1)


for i, path in enumerate(cond_sub_base_path):
    plt.figure()
    cond_sub_base = np.load(path)
    # 大于50
    cond_sub_base = cond_sub_base[(cond_sub_base > 50) & (cond_sub_base < vmax)]
    data = cond_sub_base.flatten()
    hist_data, bin_edges, _ = plt.hist(data, bins=bin_edges, alpha=0.7, edgecolor='none', label=f"target = {states[i]}us")

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    if states[i]<850:
        params, covariance = curve_fit(bimodal, bin_centers, hist_data, p0=initial_guess[i])
        mu1, sigma1, mu2, sigma2, A1, A2 = params

        # 绘制数据和拟合曲线
        x = np.linspace(0, vmax, vmax)
        plt.hist(bin_centers, weights=hist_data, bins=bin_edges, density=True, alpha=0.6, color='g')
        plt.plot(x, bimodal(x, mu1, sigma1, mu2, sigma2, A1, A2), 'r-', lw=2)
        plt.title(f'target={states[i]}us\nleft:mean={mu1:.2f},sigma={sigma1:.2f},A1={A1:.2f}\nright:mean={mu2:.2f},sigma={sigma2:.2f},A2={A2:.2f}')
        plt.xlabel('cond(uS)')
        plt.ylabel('Probability Density')
        plt.savefig(root_path+f"/hist/{states[i]}us")
        plt.show()
    else:
        params, covariance = curve_fit(normal_distribution, bin_centers, hist_data, p0=[states[i],100,1])
        mu1, sigma1,A1 = params

        # 绘制数据和拟合曲线
        x = np.linspace(0, vmax, vmax)
        plt.hist(bin_centers, weights=hist_data, bins=bin_edges, density=True, alpha=0.6, color='g')
        plt.plot(x, normal_distribution(x, mu1, sigma1, A1), 'r-', lw=2)
        plt.title(f'target={states[i]}\nmean={mu1:.2f},sigma={sigma1:.2f},A1={A1:.2f}')
        plt.xlabel('cond(uS)')
        plt.ylabel('Probability Density')
        plt.savefig(root_path+f"/hist/{states[i]}us")
        plt.show()

    # 打印拟合参数
    # print(f"mu1: {mu1}, sigma1: {sigma1}, A1: {A1}")
    # print(f"mu2: {mu2}, sigma2: {sigma2}, A2: {A2}")


    # plt.xlabel("cond(uS)")
    # plt.ylabel("Frequency")
    # plt.title(f"{states[i]}us")

    # plt.savefig(root_path+f"{states[i]}us")
    # plt.show()

In [ ]:
states = np.array([i*25+225 for i in range(32)])
root_path = "./result/4bit精度多次写验证结果/"
write_times = 40

cond_sub_base_path = []
for i in states:
    cond_sub_base_path.append(root_path+f"{i}us/after_write_verify_time={write_times}.npy")


# 初始化正态分布的预测情况
initial_guess = [
    # [225, 100, 900, 250, 1, 1],
]
for i in range(32):
    initial_guess.append([states[i],100,900,250,1,1])

def bimodal(x, mu1, sigma1, mu2, sigma2, A1, A2):
    return A1 * norm.pdf(x, mu1, sigma1) + A2 * norm.pdf(x, mu2, sigma2)

def normal_distribution(x, mu, sigma, A):
    return A * norm.pdf(x, mu, sigma)

# 设置直方图的区间
interval = 1
vmax = 1500
bin_edges = np.linspace(0, vmax, int(vmax / interval) + 1)


for i, path in enumerate(cond_sub_base_path):
    plt.figure()
    cond_sub_base = np.load(path)
    # 大于50
    cond_sub_base = cond_sub_base[(cond_sub_base > 50) & (cond_sub_base < vmax)]
    data = cond_sub_base.flatten()
    hist_data, bin_edges, _ = plt.hist(data, bins=bin_edges, alpha=0.7, edgecolor='none', label=f"target = {states[i]}us")

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    params, covariance = curve_fit(normal_distribution, bin_centers, hist_data, p0=[states[i],100,1])
    mu1, sigma1,A1 = params

    # 绘制数据和拟合曲线
    x = np.linspace(0, vmax, vmax)
    plt.hist(bin_centers, weights=hist_data, bins=bin_edges, density=True, alpha=0.6, color='g')
    plt.plot(x, normal_distribution(x, mu1, sigma1, A1), 'r-', lw=2)
    plt.title(f'target={states[i]}\nmean={mu1:.2f},sigma={sigma1:.2f},A1={A1:.2f}')
    plt.xlabel('cond(uS)')
    plt.ylabel('Probability Density')
    plt.savefig(root_path+f"/hist_write40/{states[i]}us")
    plt.show()

    # 打印拟合参数
    # print(f"mu1: {mu1}, sigma1: {sigma1}, A1: {A1}")
    # print(f"mu2: {mu2}, sigma2: {sigma2}, A2: {A2}")


    # plt.xlabel("cond(uS)")
    # plt.ylabel("Frequency")
    # plt.title(f"{states[i]}us")

    # plt.savefig(root_path+f"{states[i]}us")
    # plt.show()

# 0.数据处理

In [ ]:
def show_hist(states,root_path,write_times,good_device=None):
    cond_sub_base_path = []
    for i in states:
        cond_sub_base_path.append(root_path+f"{i}us/after_write_verify_time={write_times}.npy")

    # 设置直方图的区间
    interval = 1
    bin_edges = np.linspace(0, 1200, int(1200 / interval) + 1)

    plt.figure(figsize=(12,10))

    for i, path in enumerate(cond_sub_base_path):
        cond_sub_base = np.load(path)
        if good_device is not None:
            cond_sub_base = cond_sub_base[good_device]
        data = cond_sub_base.flatten()
        counts, bin_edges, _ = plt.hist(data, bins=bin_edges, alpha=0.7, edgecolor='none', label=f"target = {states[i]}us")

    # plt.legend()
    plt.xlabel("cond(uS)")
    plt.ylabel("Frequency")
    # plt.title(f"write = {write_times}")

    plt.savefig(root_path+f"write = {write_times}.png")
    plt.show()

def show_hist2(states,root_path,write_times,good_device=None):
    cond_sub_base_path = []
    for i in states:
        cond_sub_base_path.append(root_path+f"{i}us/after_write_verify_time={write_times}.npy")

    # 设置直方图的区间
    interval = 1
    bin_edges = np.linspace(0, 1200, int(1200 / interval) + 1)

    for i, path in enumerate(cond_sub_base_path):
        plt.figure()
        cond_sub_base = np.load(path)
        if good_device is not None:
            cond_sub_base = cond_sub_base[good_device]
        data = cond_sub_base.flatten()
        counts, bin_edges, _ = plt.hist(data, bins=bin_edges, alpha=0.7, edgecolor='none', label=f"target = {states[i]}us")

        plt.xlabel("cond(uS)")
        plt.ylabel("Frequency")
        plt.title(f"{states[i]}us")

        plt.savefig(root_path+f"{states[i]}us")
        plt.show()

def show_cdf(states,root_path,write_times,good_device=None):
    cond_sub_base_path = []
    for i in states:
        cond_sub_base_path.append(root_path+f"{i}us/after_write_verify_time={write_times}.npy")

    plt.figure(figsize=(12,10))
    for i, path in enumerate(cond_sub_base_path):
        cond_sub_base = np.load(path)
        if good_device is not None:
            cond_sub_base = cond_sub_base[good_device]
        data = cond_sub_base.flatten()
        sorted_data = np.sort(data)
        # 计算累积比例
        cdf = np.linspace(1/len(sorted_data), 1, len(sorted_data))
        plt.plot(sorted_data, cdf, marker='.', linestyle='none')

    # plt.legend()
    plt.xlabel("cond(uS)")
    plt.ylabel("CDF")
    # plt.title(f"write = {write_times}")

    plt.savefig(root_path+f"write = {write_times}.png")
    plt.show()

def get_good_device(states,root_path,write_times,threshold,in_times):
    good_device = np.zeros((256,256))
    for i in states:
        filepath=root_path+f"{i}us/after_write_verify_time={write_times}.npy"
        # print(filepath)
        cond_sub_base = np.load(filepath)
        good_device +=(cond_sub_base>(i-threshold))&(cond_sub_base<(i+threshold))
    return good_device>=in_times

In [ ]:
states32 = np.array([i*25+225 for i in range(32)])
states16_1 = np.array([i*50+250 for i in range(16)])
states16_2 = np.array([i*50+225 for i in range(16)])
states8_1 = np.array([i*100+300 for i in range(8)])
states8_2 = np.array([i*100+250 for i in range(8)])
print(states16_2)

In [ ]:
root_path32 = "./result/4bit精度多次写验证结果/"
show_hist2(states32,root_path32,1)

In [ ]:
# root_path32 = "./result/3bit精度多次写验证结果/"
# show_hist2(states16_1,root_path32,40)

In [ ]:
root_path32 = "./result/5bit精度多次写验证结果/"
good_device32 = get_good_device(states32,root_path32,40,50,32)
print(np.sum(good_device32))
print(good_device32.shape)

In [ ]:
show_hist(states32,root_path32,40,good_device32)

In [ ]:
show_cdf(states32,root_path32,40,good_device32)

In [ ]:
root_path16 = "./result/5bit精度多次写验证结果/"
good_device16 = get_good_device(states16_1,root_path16,40,25,16)
print(np.sum(good_device16))
print(good_device16.shape)

In [ ]:
show_hist(states16_1,root_path16,40,good_device16)
show_cdf(states16_1,root_path16,40,good_device16)

In [ ]:
root_path16 = "./result/4bit精度多次写验证结果/"
good_device16 = get_good_device(states8_1,root_path16,40,50,8)
print(np.sum(good_device16))

In [ ]:
show_hist(states8_1,root_path16,40,good_device16)
show_cdf(states8_1,root_path16,40,good_device16)

In [ ]:
root_path16 = "./result/4bit精度多次写验证结果/"
good_device16 = get_good_device(states8_1,root_path16,40,50,8)
print(np.sum(good_device16))

In [ ]:
show_hist(states8_1,root_path16,40,good_device16)
show_cdf(states8_1,root_path16,40,good_device16)

# 取区域数据进行处理

In [ ]:
def show_hist(states,root_path,write_times,threshold,datatype=0):
    cond_sub_base_path = []
    for i in states:
        cond_sub_base_path.append(root_path+f"{i}us/after_write_verify_time={write_times}.npy")

    # 设置直方图的区间
    interval = 1
    bin_edges = np.linspace(0, 1200, int(1200 / interval) + 1)

    plt.figure(figsize=(12,10))

    interval = int(256/len(states))
    points_num = 0
    for i, path in enumerate(cond_sub_base_path):
        if datatype==0:
            cond_sub_base = np.load(path)
        elif datatype==1:
            cond_sub_base = np.load(path)[i*interval:(i+1)*interval,:]
        else:
            cond_sub_base = np.load(path)[:,i*interval:(i+1)*interval]
        data = cond_sub_base.flatten()
        data = data[(data>=(states[i]-threshold))&(data<=(states[i]+threshold))]
        points_num +=len(data)
        counts, bin_edges, _ = plt.hist(data, bins=bin_edges, alpha=0.7, edgecolor='none', label=f"target = {states[i]}us")

    # plt.legend()
    plt.xlabel("cond(uS)")
    plt.ylabel("Frequency")
    # plt.title(f"write = {write_times}")

    plt.savefig(root_path+f"write = {write_times}.png")
    plt.show()
    return points_num

def show_cdf(states,root_path,write_times,threshold,datatype=0):
    cond_sub_base_path = []
    for i in states:
        cond_sub_base_path.append(root_path+f"{i}us/after_write_verify_time={write_times}.npy")

    plt.figure(figsize=(12,10))
    points_num = 0
    interval = int(256/len(states))
    for i, path in enumerate(cond_sub_base_path):
        if datatype==0:
            cond_sub_base = np.load(path)
        elif datatype==1:
            cond_sub_base = np.load(path)[i*interval:(i+1)*interval,:]
        else:
            cond_sub_base = np.load(path)[:,i*interval:(i+1)*interval]
        data = cond_sub_base.flatten()
        data = data[(data>=(states[i]-threshold))&(data<=(states[i]+threshold))]
        points_num +=len(data)

        sorted_data = np.sort(data)
        # 计算累积比例
        cdf = np.linspace(1/len(sorted_data), 1, len(sorted_data))
        plt.plot(sorted_data, cdf, marker='.', linestyle='none')

    # plt.legend()
    plt.xlabel("cond(uS)")
    plt.ylabel("CDF")
    # plt.title(f"write = {write_times}")

    plt.savefig(root_path+f"write = {write_times}.png")
    plt.show()
    return points_num

In [ ]:
states32 = np.array([i*25+225 for i in range(32)])
states16_1 = np.array([i*50+250 for i in range(16)])
states16_2 = np.array([i*50+225 for i in range(16)])
states8_1 = np.array([i*100+300 for i in range(8)])
states8_2 = np.array([i*100+250 for i in range(8)])

In [ ]:
root_path32 = "./result/4bit精度多次写验证结果/"
show_hist(states32,root_path32,40,threshold=12.5,datatype=2)
show_cdf(states32,root_path32,40,threshold=12.5,datatype=2)

In [ ]:
root_path16 = "./result/4bit精度多次写验证结果/"
show_hist(states16_1,root_path16,40,threshold=25,datatype=2)
show_cdf(states16_1,root_path16,40,threshold=25,datatype=2)

show_hist(states16_2,root_path16,40,threshold=25,datatype=2)
show_cdf(states16_2,root_path16,40,threshold=25,datatype=2)

In [ ]:
root_path8 = "./result/3bit精度多次写验证结果/"
a=show_hist(states8_1,root_path8,40,threshold=50,datatype=2)
show_cdf(states8_1,root_path8,40,threshold=50,datatype=2)

b=show_hist(states8_2,root_path8,40,threshold=50,datatype=2)
show_cdf(states8_2,root_path8,40,threshold=50,datatype=2)
print(a,b)

# 1.拟合tg和cond的线性映射

In [ ]:
# 3v,set脉宽1us,reset脉宽1us
tg_map = [1.4,1.5,1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3,2.4]
cond_map = [190,310,430,510,610,710,850,910,1030,1110,1190]
# 3v,set脉宽1us,reset脉宽10us
# tg_map = [1.4,1.5,1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3,2.4]
# cond_map = [190,310,430,510,610,710,850,910,1030,1110,1190]

# 
# tg_map = [1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3]
# cond_map = [550,670,790,890,1010,1090,1170,1250]

tg_map = np.array(tg_map)
cond_map = np.array(cond_map)

coefficients = np.polyfit(tg_map, cond_map, 1)  # 返回斜率和截距
slope = coefficients[0]
intercept = coefficients[1]

y_fit = slope * tg_map + intercept

plt.scatter(tg_map, cond_map, color='blue', label='Data points')  # 原始数据点
plt.plot(tg_map, y_fit, color='red', label=f'cond = {slope:.2f} * tg + {intercept:.2f}')  # 拟合直线
plt.xlabel('tg(v)')
plt.ylabel('cond(us)')
plt.legend()
plt.title('Linear Fit')
plt.show()

print(intercept,slope)

# 2.写对应电导值

In [ ]:
def write_verify(write_time,tg_v,target_cond,threshold,set_pulse_width,reset_pulse_width,root_path):
    print(f"目标电导{target_cond},初始tg电压{tg_v:.2f},写验证次数{write_time},set脉宽{set_pulse_width},reset脉宽{reset_pulse_width}")
    if not os.path.exists(root_path):
        os.makedirs(root_path)
    tg_v = np.ones((256,256))*tg_v
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    cond_sub_base = None
    vmin = 0
    vmax = 1400
    # 先统一reset
    chip.write_point2(crossbar=np.ones((256,256)),write_voltage=3,tg=5,pulse_width=reset_pulse_width,set_device=False)
    for i in range(write_time):
        print(f"第{i}次写验证")
        voltage_base = chip.read_point3(0,256,0,256,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
        voltage = chip.read_point3(0,256,0,256,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        np.save(root_path+f"after_write_verify_time={int(i)}.npy", cond_sub_base)

        condition_reset = cond_sub_base > (target_cond+threshold)
        condition_set = cond_sub_base < (target_cond-threshold)
        
        path = root_path+f"{i}.png"
        plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)

        if i>0:
            if i<5:
                tg_v[condition_reset] -= 0.08
                tg_v[condition_set] += 0.08
            elif i<10:
                tg_v[condition_reset] -= 0.04
                tg_v[condition_set] += 0.04
            else:
                tg_v[condition_reset] -= 0.02
                tg_v[condition_set] += 0.02
            # 避免电压超过范围
            tg_v.clip(0, 5, out=tg_v)
        # reset的点
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=5,pulse_width=reset_pulse_width,set_device=False)
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=tg_v,pulse_width=set_pulse_width,set_device=True)

        # set的点
        chip.write_point2(crossbar=condition_set,write_voltage=3,tg=tg_v,pulse_width=set_pulse_width,set_device=True)


    voltage_base = chip.read_point3(0,256,0,256,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point3(0,256,0,256,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    np.save(root_path+f"after_write_verify_time={write_time}.npy", cond_sub_base)
    np.save(root_path+f"tg_v.npy", tg_v)


    condition_reset = cond_sub_base > (target_cond+threshold)
    condition_set = cond_sub_base < (target_cond-threshold)
    path = root_path+f"{i}.png"
    plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)
    return cond_sub_base

In [ ]:
for i in states32:
    tg=(i-intercept)/slope
    write_verify(write_time=40,tg_v=tg,target_cond=i,threshold=12.5,
                 set_pulse_width=1e-6,reset_pulse_width=1e-6,root_path=f"./result/5bit精度多次写验证结果/{i}us/")

In [ ]:
for i in states16_1:
    tg=(i-intercept)/slope
    write_verify(write_time=40,tg_v=tg,target_cond=i,threshold=25,
                 set_pulse_width=1e-6,reset_pulse_width=1e-6,root_path=f"./result/4bit精度多次写验证结果/{i}us/")
for i in states16_1:
    tg=(i-intercept)/slope
    write_verify(write_time=40,tg_v=tg,target_cond=i,threshold=25,
                 set_pulse_width=1e-6,reset_pulse_width=1e-6,root_path=f"./result/4bit精度多次写验证结果/{i}us/")

In [ ]:
for i in states8_1:
    tg=(i-intercept)/slope
    write_verify(write_time=40,tg_v=tg,target_cond=i,threshold=50,
                 set_pulse_width=1e-6,reset_pulse_width=1e-6,root_path=f"./result/3bit精度多次写验证结果/{i}us/")
for i in states8_2:
    tg=(i-intercept)/slope
    write_verify(write_time=40,tg_v=tg,target_cond=i,threshold=50,
                 set_pulse_width=1e-6,reset_pulse_width=1e-6,root_path=f"./result/3bit精度多次写验证结果/{i}us/")

In [ ]:
print(chip.ps.receive_packet(8906))

In [ ]:
def write_verify(start_time,write_time,tg_v,target_cond,threshold,set_pulse_width,reset_pulse_width,root_path):
    print(f"目标电导{target_cond},初始tg电压{tg_v:.2f},写验证次数{write_time},set脉宽{set_pulse_width},reset脉宽{reset_pulse_width}")
    if not os.path.exists(root_path):
        os.makedirs(root_path)
    tg_v = np.ones((256,256))*tg_v
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    cond_sub_base = None
    vmin = 0
    vmax = 1400
    need_read = np.ones((256,256))
    # 先统一reset
    if start_time ==0:
        chip.write_point2(crossbar=need_read,write_voltage=3,tg=5,pulse_width=reset_pulse_width,set_device=False)
    for i in range(start_time,write_time):
        print(f"第{i}次写验证")
        voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
        voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        np.save(root_path+f"after_write_verify_time={int(i)}.npy", cond_sub_base)

        condition_reset = cond_sub_base > (target_cond+threshold)
        condition_set = cond_sub_base < (target_cond-threshold)
        
        path = root_path+f"{i}.png"
        plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)

        if i>0:
            if i<5:
                tg_v[condition_reset] -= 0.08
                tg_v[condition_set] += 0.08
            elif i<10:
                tg_v[condition_reset] -= 0.04
                tg_v[condition_set] += 0.04
            else:
                tg_v[condition_reset] -= 0.02
                tg_v[condition_set] += 0.02
            # 避免电压超过范围
            tg_v.clip(0, 5, out=tg_v)
        # reset的点
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=5,pulse_width=reset_pulse_width,set_device=False)
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=tg_v,pulse_width=set_pulse_width,set_device=True)

        # set的点
        chip.write_point2(crossbar=condition_set,write_voltage=3,tg=tg_v,pulse_width=set_pulse_width,set_device=True)


    voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    np.save(root_path+f"after_write_verify_time={write_time}.npy", cond_sub_base)
    np.save(root_path+f"tg_v.npy", tg_v)


    condition_reset = cond_sub_base > target_cond
    condition_set = cond_sub_base < target_cond
    path = root_path+f"{i}.png"
    plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)
    return cond_sub_base

In [ ]:
states = np.array([i*25+225 for i in range(36)])
print(states)
print(len(states))
print((states-intercept)/slope)

In [ ]:
for i in states:
    tg=(i-intercept)/slope
    write_verify(start_time=0, write_time=50,tg_v=tg,target_cond=i,threshold=25,
                 set_pulse_width=1e-6,reset_pulse_width=1e-6,root_path=f"./result/5bit精度多次写验证结果/{i}us/")

In [ ]:
chip.ps.receive_packet(8096)

# 3. 验证比特精度

In [ ]:
def good_device_write_verify(write_time,good_device,tg_v,cond_upper,cond_lower,set_pulse_width,reset_pulse_width,root_path):
    need_read = good_device
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    cond_sub_base = None
    vmin = cond_lower
    vmax = cond_upper
    chip.write_point2(crossbar=need_read,write_voltage=3,tg=5,pulse_width=reset_pulse_width,set_device=False)
    for i in range(write_time):
        print(f"第{i}次写验证")
        voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        np.save(root_path+f"after_write_verify_time={int(i)}.npy", cond_sub_base)

        condition_reset = (cond_sub_base > cond_upper)&need_read
        condition_set = (cond_sub_base < cond_lower)&need_read
        # need_read = condition_reset|condition_set
        
        path = root_path+f"{i}.png"
        # voltage_base[good_device] = chip.read_point2(crossbar=good_device, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[good_device]
        # voltage[good_device] = chip.read_point2(crossbar=good_device, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[good_device]
        # cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)

        if i>0:
            tg_v[condition_reset] -= 0.02
            tg_v[condition_set] += 0.02
        # reset的点
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=5,pulse_width=reset_pulse_width,set_device=False)
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=tg_v,pulse_width=set_pulse_width,set_device=True)

        # set的点
        chip.write_point2(crossbar=condition_set,write_voltage=3+i*0.025,tg=tg_v,pulse_width=set_pulse_width,set_device=True)

    voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    np.save(root_path+f"after_write_verify_time={write_time}.npy", cond_sub_base)
    np.save(root_path+f"tg_v.npy", tg_v)
    np.save(root_path+f"good_device.npy", good_device)


    condition_reset = cond_sub_base > cond_upper
    condition_set = cond_sub_base < cond_lower
    path = root_path+f"{i}.png"
    plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)
    return cond_sub_base

In [ ]:
# voltage_base = chip.read_point2(crossbar=np.ones((256,256)), read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
# voltage = chip.read_point2(crossbar=np.ones((256,256)), read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
# cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)

# np.save("./data/conds.npy",cond_sub_base)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import curve_fit

# 生成示例数据
np.random.seed(0)
data1 = np.random.normal(loc=50, scale=10, size=1000)
data2 = np.random.normal(loc=200, scale=20, size=1000)
data = np.concatenate([data1, data2])

# 定义双正态分布函数
def bimodal(x, mu1, sigma1, mu2, sigma2, A1, A2):
    return A1 * norm.pdf(x, mu1, sigma1) + A2 * norm.pdf(x, mu2, sigma2)

# 初始参数猜测
initial_guess = [50, 10, 200, 20, 1, 1]

# 拟合数据
params, covariance = curve_fit(bimodal, xdata=data, ydata=data, p0=initial_guess)

# 提取拟合参数
mu1, sigma1, mu2, sigma2, A1, A2 = params

# 绘制数据和拟合曲线
x = np.linspace(0, 300, 1000)
plt.hist(data, bins=50, density=True, alpha=0.6, color='g')
plt.plot(x, bimodal(x, mu1, sigma1, mu2, sigma2, A1, A2), 'r-', lw=2)
plt.title('Bimodal Normal Distribution Fit')
plt.xlabel('Value')
plt.ylabel('Probability Density')
plt.show()

# 打印拟合参数
print(f"mu1: {mu1}, sigma1: {sigma1}, A1: {A1}")
print(f"mu2: {mu2}, sigma2: {sigma2}, A2: {A2}")

In [ ]:
slopes_cond_to_tg = np.load("./data/slopes_cond_to_tg.npy")
intercepts_cond_to_tg = np.load("./data/intercepts_cond_to_tg.npy")
slopes_tg_to_cond = np.load("./data/slopes_tg_to_cond.npy")
intercepts_tg_to_cond = np.load("./data/intercepts_tg_to_cond.npy")
good_device = np.load('./data/good_device.npy')
conds = np.load("./data/conds.npy")
good_device = (slopes_tg_to_cond>800)&(slopes_tg_to_cond<1200)&(conds<200)&good_device
print(np.sum(good_device))

In [ ]:
target = 175
cond_min,cond_max = 200,1000
state_num = 16
threshold = (cond_max-cond_min)/state_num/2
for i in range(15):
    target = target+50
    img_tg = target*slopes_cond_to_tg + intercepts_cond_to_tg
    good_device_write_verify(write_time=30,good_device=good_device,tg_v=img_tg,
                             cond_upper = target+threshold,cond_lower=target-threshold,
                             set_pulse_width = 1e-6,reset_pulse_width=1e-6,root_path=f"./result/4bit精度2/{target}us/")

In [ ]:
target = 200
cond_min,cond_max = 200,1000
state_num = 32
threshold = (cond_max-cond_min)/state_num/2
for i in range(15):
    target = target+50
    img_tg = target*slopes_cond_to_tg + intercepts_cond_to_tg
    good_device_write_verify(write_time=30,good_device=good_device,tg_v=img_tg,
                                cond_upper = target+threshold,cond_lower=target-threshold,
                                set_pulse_width = 1e-6,reset_pulse_width=1e-6,root_path=f"./result/5bit精度/{target}us/")

In [ ]:
img_tg = np.ones((256,256))*(10*100 - intercept)/slope-0.1
write_verify(40,img_tg,10*100,1e-6,1e-6,root_path=f"./result/精度/{10*100}us/")

In [ ]:
# def write_verify(write_time,tg_v,target_cond,set_pulse_width,reset_pulse_width,root_path):

In [ ]:
for i in range(2,12):
    img_tg = np.ones((256,256))*(i*100 - intercept)/slope
    write_verify(40,img_tg,i*100,1e-6,1e-6,root_path=f"./result/精度/{i*100}us/")

# 3. 直方图绘制

In [ ]:
target = 225
cond_sub_base_path = []
root_path=f"./result/5bit精度/{target}us/"
for i in range(20):
    cond_sub_base_path.append(root_path+f"after_write_verify_time={int(i)}.npy")
print(cond_sub_base_path)

In [ ]:
interval = 5
bin_edges = np.linspace(0, 2000, int(2000/interval)+1)  
for i,path in enumerate(cond_sub_base_path):
    cond_sub_base = np.load(path)[good_device]
    data = cond_sub_base.flatten()
    counts, bin_edges, _ = plt.hist(data, bins=bin_edges, color='blue', alpha=0.7, edgecolor='black')
    plt.title(f"target = {target},write = {i}")
    plt.xlabel("cond(uS)")
    plt.ylabel("Frequency")
    path = root_path+f"target = {target},write = {i}.png"
    plt.savefig(path)
    plt.show()